# Twisted Bilayer Graphene — MPO Hamiltonian

Builds a TBG tight-binding Hamiltonian as an MPO and computes the density of states via KPM.

`twisted_bilayer_hamiltonian` constructs the full system in one call:
- Intra-layer NN hopping via `build_hamiltonian`
- Interlayer coupling $V_{ij} = t_\perp e^{-\alpha |\mathbf{r}_i^{(1)} - \mathbf{r}_j^{(2)}|}$ via QTCI
- Rotation of layer 2 around the lattice centroid

The returned `TBHamiltonian` plugs directly into `KPM_Tn` for spectral calculations.

In [ ]:
using LinearAlgebra
using Plots
using ITensors
using ITensorMPS
include("../src/TensorBinding.jl")
using .TensorBinding

# ── Parameters ────────────────────────────────────────────────────────────────
Lx    = 3          # 2^Lx = 8 sites per row
Ly    = 3          # 2^Ly = 8 rows  →  N = 64 sites per layer
θ_deg = 5.0        # twist angle (degrees)

# ── Build the TBG Hamiltonian in one call ─────────────────────────────────────
H = TensorBinding.twisted_bilayer_hamiltonian(:honeycomb, Lx, Ly, θ_deg;
        t_intra = 1.0,
        t_inter = 0.3,
        α_decay = 1 / 1.5)

println("Sites per layer : $(2^(Lx+Ly))")
println("MPO length      : $(length(H.mpo))")
println("Max bond dim    : $(ITensorMPS.maxlinkdim(H.mpo))")

In [ ]:
# ── KPM density of states ─────────────────────────────────────────────────────
Ncheb      = 150
maxdim_kpm = 200

Tn, scale, center = TensorBinding.KPM_Tn(H, Ncheb; maxdim=maxdim_kpm, cutoff=1e-5)

ω_grid = range(center - scale, center + scale; length=200)
dos = [begin
    ω_r = (ω - center) / scale
    abs(ω_r) >= 1.0 ? 0.0 :
        real(tr(TensorBinding.get_ldos_w_from_Tn(Tn, Ncheb, ω_r; maxdim=maxdim_kpm)))
end for ω in ω_grid]

plot(ω_grid, dos;
     xlabel="Energy (t)", ylabel="DoS",
     title="TBG DoS  (θ=$(θ_deg)°,  $(2*2^(Lx+Ly)) sites)",
     legend=false, lw=2)

In [ ]:
# ── DoS vs twist angle ────────────────────────────────────────────────────────
θ_sweep   = range(0.0, 10.0; length=8)
ω_sweep   = range(center - scale, center + scale; length=150)
dos_sweep = Matrix{Float64}(undef, length(ω_sweep), length(θ_sweep))

for (idx, θ_d) in enumerate(θ_sweep)
    H_loc = TensorBinding.twisted_bilayer_hamiltonian(:honeycomb, Lx, Ly, θ_d;
                t_intra=1.0, t_inter=0.3, α_decay=1/1.5)
    Tn_loc, sc, cn = TensorBinding.KPM_Tn(H_loc, Ncheb;
                         maxdim=maxdim_kpm, cutoff=1e-5)
    for (iω, ω) in enumerate(ω_sweep)
        ω_r = (ω - cn) / sc
        dos_sweep[iω, idx] = abs(ω_r) >= 1.0 ? 0.0 :
            real(tr(TensorBinding.get_ldos_w_from_Tn(Tn_loc, Ncheb, ω_r; maxdim=maxdim_kpm)))
    end
    println("θ = $(round(θ_d; digits=2))° done")
end

heatmap(collect(θ_sweep), collect(ω_sweep), dos_sweep;
        xlabel="Twist angle θ (°)", ylabel="Energy (t)",
        title="TBG DoS vs twist angle",
        color=:inferno, colorbar_title="DoS")

---
## Exact diagonalization benchmark

Build the same $2N \times 2N$ Hamiltonian as a dense matrix and compare the KPM DoS against
Gaussian-broadened exact eigenvalues.

In [ ]:
N   = 2^(Lx + Ly)
t   = 1.0
rs1 = TensorBinding.lattice_positions(:honeycomb, Lx, Ly; angle_deg=0.0)
rs2 = TensorBinding.lattice_positions(:honeycomb, Lx, Ly; angle_deg=θ_deg)

H_mono_dense = [abs(norm(rs1[i,:] - rs1[j,:]) - 1.0) < 1e-6 ? t : 0.0
                for i in 1:N, j in 1:N]
V_dense      = [0.3 * exp(-(1/1.5) * norm(rs1[i,:] - rs2[j,:])) for i in 1:N, j in 1:N]
H_dense      = [H_mono_dense  V_dense; V_dense'  H_mono_dense]

println("Hermitian: $(ishermitian(H_dense))")
println("NN bonds per site: $(round(sum(H_mono_dense .!= 0) / N; digits=2))  (expect 3)")

In [ ]:
σ_broad  = 0.05
ε_exact  = eigvals(Hermitian(H_dense))
dos_exact = [sum(@. exp(-(ω - ε_exact)^2 / (2σ_broad^2)) / (sqrt(2π) * σ_broad))
             for ω in ω_grid] ./ (2N)

dos_kpm_norm   = dos      ./ maximum(dos)
dos_exact_norm = dos_exact ./ maximum(dos_exact)

plot(ω_grid, dos_kpm_norm;
     label="KPM (Ncheb=$(Ncheb))", lw=2, color=:firebrick,
     xlabel="Energy (t)", ylabel="DoS (normalised)",
     title="TBG DoS — KPM vs exact  (θ=$(θ_deg)°)")
plot!(ω_grid, dos_exact_norm;
      label="Exact diag (σ=$(σ_broad) t)", lw=2, ls=:dash, color=:steelblue)